In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
#from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr


In [2]:
# pipeline_submission.py
# Versión corregida + búsqueda de hiperparámetros + features ampliadas + revisión final

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr
from catboost import CatBoostClassifier, CatBoostRegressor

RANDOM_STATE = 42
NFOLDS = 5

# Hiperparámetros
RUN_HYPERPARAM_SEARCH = True
TUNE_SAMPLE_FRAC = 0.35
N_ITER_CLF = 12
N_ITER_REG = 12
TUNE_FOLDS = 3

TRANS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/transactions_2016_2017.csv"
TRAIN_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/customer_clv_train.csv"
TEST_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/customer_clv_test.csv"
OUTPUT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/submission.csv"


def make_strat_labels(y, n_bins=10):
    y = np.asarray(y)
    labels = np.zeros(len(y), dtype=int)
    pos_mask = y > 0
    if pos_mask.sum() == 0:
        return labels
    pos_values = pd.Series(y[pos_mask])
    q = min(n_bins, max(2, pos_mask.sum()))
    try:
        pos_bins = pd.qcut(pos_values, q=q, labels=False, duplicates="drop")
        labels[pos_mask] = pos_bins.to_numpy(dtype=int) + 1
    except ValueError:
        labels[pos_mask] = 1
    return labels


def add_window_features(trx: pd.DataFrame, customer_features: pd.DataFrame, reference_date: pd.Timestamp):
    windows = [30, 90, 180, 365]

    # base table with one row per customer
    if "cust_id" not in customer_features.columns:
        return customer_features

    for w in windows:
        cutoff = reference_date - pd.Timedelta(days=w)
        w_trx = trx[trx["order_date"] >= cutoff].copy()

        if len(w_trx) == 0:
            tmp = pd.DataFrame({"cust_id": customer_features["cust_id"]})
        else:
            tmp = w_trx.groupby("cust_id").agg(
                **{
                    f"revenue_{w}d": ("sale_revenue", "sum"),
                    f"orders_{w}d": ("sale_id", "nunique") if "sale_id" in w_trx.columns else ("sale_revenue", "size"),
                    f"lines_{w}d": ("sale_revenue", "size"),
                    f"avg_ticket_{w}d": ("sale_revenue", "mean"),
                }
            ).reset_index()

        customer_features = customer_features.merge(tmp, on="cust_id", how="left")

    # trend features
    last_6m = trx[trx["order_date"] >= (reference_date - pd.Timedelta(days=180))]
    prev_6m = trx[(trx["order_date"] < (reference_date - pd.Timedelta(days=180))) &
                  (trx["order_date"] >= (reference_date - pd.Timedelta(days=360)))]

    last_3m = trx[trx["order_date"] >= (reference_date - pd.Timedelta(days=90))]
    prev_3m = trx[(trx["order_date"] < (reference_date - pd.Timedelta(days=90))) &
                  (trx["order_date"] >= (reference_date - pd.Timedelta(days=180)))]

    rev_last_6m = last_6m.groupby("cust_id")["sale_revenue"].sum().rename("revenue_last_6m")
    rev_prev_6m = prev_6m.groupby("cust_id")["sale_revenue"].sum().rename("revenue_prev_6m")
    ord_last_3m = (last_3m.groupby("cust_id")["sale_id"].nunique().rename("orders_last_3m")
                   if "sale_id" in trx.columns else last_3m.groupby("cust_id").size().rename("orders_last_3m"))
    ord_prev_3m = (prev_3m.groupby("cust_id")["sale_id"].nunique().rename("orders_prev_3m")
                   if "sale_id" in trx.columns else prev_3m.groupby("cust_id").size().rename("orders_prev_3m"))

    trend = pd.concat([rev_last_6m, rev_prev_6m, ord_last_3m, ord_prev_3m], axis=1).reset_index()
    customer_features = customer_features.merge(trend, on="cust_id", how="left")

    eps = 1e-6
    customer_features["revenue_trend_ratio_6m"] = (
        customer_features["revenue_last_6m"] / (customer_features["revenue_prev_6m"] + eps)
    )
    customer_features["orders_trend_diff_3m"] = (
        customer_features["orders_last_3m"] - customer_features["orders_prev_3m"]
    )

    return customer_features


def add_cadence_features(trx: pd.DataFrame, customer_features: pd.DataFrame):
    if "sale_id" in trx.columns:
        ord_dates = trx.groupby(["cust_id", "sale_id"], as_index=False)["order_date"].min()
    else:
        ord_dates = trx[["cust_id", "order_date"]].copy()

    ord_dates = ord_dates.sort_values(["cust_id", "order_date"])
    ord_dates["prev_order_date"] = ord_dates.groupby("cust_id")["order_date"].shift(1)
    ord_dates["interpurchase_days"] = (ord_dates["order_date"] - ord_dates["prev_order_date"]).dt.days

    cadence = ord_dates.groupby("cust_id")["interpurchase_days"].agg(
        interpurchase_mean="mean",
        interpurchase_std="std",
        interpurchase_max="max",
    ).reset_index()

    customer_features = customer_features.merge(cadence, on="cust_id", how="left")
    return customer_features


def add_diversity_features(trx: pd.DataFrame, customer_features: pd.DataFrame):
    grp = trx.groupby("cust_id")

    diversity = grp.agg(
        n_unique_brands=("prod_brand", "nunique") if "prod_brand" in trx.columns else ("sale_revenue", "size"),
        n_unique_type1=("prod_type_1", "nunique") if "prod_type_1" in trx.columns else ("sale_revenue", "size"),
        n_unique_type3=("prod_type_3", "nunique") if "prod_type_3" in trx.columns else ("sale_revenue", "size"),
    ).reset_index()

    customer_features = customer_features.merge(diversity, on="cust_id", how="left")

    # brand concentration: share of top brand
    if "prod_brand" in trx.columns:
        brand_counts = trx.groupby(["cust_id", "prod_brand"]).size().rename("cnt").reset_index()
        total_counts = brand_counts.groupby("cust_id")["cnt"].sum().rename("total_cnt")
        top_counts = brand_counts.groupby("cust_id")["cnt"].max().rename("top_brand_cnt")
        conc = pd.concat([total_counts, top_counts], axis=1).reset_index()
        conc["brand_top1_share"] = conc["top_brand_cnt"] / conc["total_cnt"].replace(0, np.nan)
        customer_features = customer_features.merge(conc[["cust_id", "brand_top1_share"]], on="cust_id", how="left")

    return customer_features


def build_customer_features(transactions: pd.DataFrame, top_brands=None) -> pd.DataFrame:
    trx = transactions.copy()
    trx["cust_id"] = trx["cust_id"].astype(str)
    trx["order_date"] = pd.to_datetime(trx["order_date"], errors="coerce")
    trx["pack_date"] = pd.to_datetime(trx.get("pack_date"), errors="coerce") if "pack_date" in trx.columns else pd.NaT

    trx["sale_revenue"] = pd.to_numeric(trx.get("sale_revenue", 0.0), errors="coerce").fillna(0.0)
    trx["sale_discount_applied"] = pd.to_numeric(trx.get("sale_discount_applied", 0.0), errors="coerce").fillna(0.0)
    trx["prod_outlet"] = pd.to_numeric(trx.get("prod_outlet", np.nan), errors="coerce")

    # SIZE FEATURES
    raw_size = trx["prod_size"].astype(str).str.strip() if "prod_size" in trx.columns else pd.Series(["unknown"] * len(trx), index=trx.index)
    size_alpha = raw_size.str.upper()
    alpha_levels = ["XS", "S", "M", "L", "XL"]
    size_alpha = np.where(size_alpha.isin(alpha_levels), size_alpha, "OTHER")

    size_alpha_dummies = pd.get_dummies(pd.Series(size_alpha, index=trx.index), prefix="size_alpha")
    size_alpha_dummies["cust_id"] = trx["cust_id"]
    size_alpha_dummies = size_alpha_dummies.groupby("cust_id").sum().reset_index()

    size_numeric = pd.to_numeric(raw_size, errors="coerce")
    bins = [0, 24, 30, 36, 40, 44, 60]
    labels = ["baby", "child", "youth", "women_small", "women_large", "men_large"]
    size_group = pd.cut(size_numeric, bins=bins, labels=labels, include_lowest=True)

    size_num_dummies = (
        size_group.astype("object")
        .fillna("unknown")
        .astype(str)
        .str.get_dummies()
        .add_prefix("size_num_")
    )
    size_num_dummies["cust_id"] = trx["cust_id"]
    size_num_dummies = size_num_dummies.groupby("cust_id").sum().reset_index()

    # BASE AGGREGATES
    customer_features = trx.groupby("cust_id").agg(
        customer_revenue=("sale_revenue", "sum"),
        customer_purchases_number=("sale_id", "nunique") if "sale_id" in trx.columns else ("sale_revenue", "size"),
        n_lines=("sale_revenue", "size"),
        mean_revenue=("sale_revenue", "mean"),
        n_products=("prod_id", "nunique") if "prod_id" in trx.columns else ("sale_revenue", "size"),
        mean_discount=("sale_discount_applied", "mean"),
        n_returned=("returned_to_shop_id", lambda x: x.notna().sum()) if "returned_to_shop_id" in trx.columns else ("sale_revenue", "size"),
        last_order_date=("order_date", "max"),
        first_order_date=("order_date", "min"),
    ).reset_index()

    reference = trx["order_date"].max()
    customer_features["recency_days"] = (reference - customer_features["last_order_date"]).dt.days
    customer_features["customer_lifespan_days"] = (customer_features["last_order_date"] - customer_features["first_order_date"]).dt.days

    # Antigüedad y actividad
    customer_features["tenure_days"] = (reference - customer_features["first_order_date"]).dt.days

    month_activity = (
        trx.assign(order_month=trx["order_date"].dt.to_period("M"))
        .groupby("cust_id")["order_month"].nunique()
        .rename("active_months")
        .reset_index()
    )
    customer_features = customer_features.merge(month_activity, on="cust_id", how="left")

    tenure_months = (customer_features["tenure_days"] / 30.44).clip(lower=1)
    customer_features["active_month_ratio"] = customer_features["active_months"] / tenure_months

    customer_features = customer_features.merge(size_alpha_dummies, on="cust_id", how="left")
    customer_features = customer_features.merge(size_num_dummies, on="cust_id", how="left")

    # TOP BRANDS (fit-safe)
    if top_brands is not None and "prod_brand" in trx.columns and len(top_brands) > 0:
        brand_counts = (
            trx[trx["prod_brand"].isin(top_brands)]
            .groupby(["cust_id", "prod_brand"])["prod_brand"]
            .count()
            .unstack(fill_value=0)
        )
        brand_counts.columns = [f"brand_{str(c)}" for c in brand_counts.columns]
        customer_features = customer_features.merge(brand_counts.reset_index(), on="cust_id", how="left")

    # MULTI LABEL
    multiple_labels = ["prod_type_3", "prod_type_4", "prod_type_5", "prod_material", "prod_print", "prod_comfort_sole", "prod_comfort_wear", "prod_clasp"]
    for col in multiple_labels:
        if col in trx.columns:
            dummies = (
                trx[col]
                .fillna("unknown")
                .astype(str)
                .str.replace(" ", "", regex=False)
                .str.get_dummies(",")
                .add_prefix(col + "_")
            )
            dummies["cust_id"] = trx["cust_id"]
            dummies = dummies.groupby("cust_id").sum().reset_index()
            customer_features = customer_features.merge(dummies, on="cust_id", how="left")

    # NORMAL CATEGORICAL
    for col in ["prod_type_1", "prod_insole"]:
        if col in trx.columns:
            dummies = (
                trx[col]
                .fillna("unknown")
                .astype(str)
                .str.get_dummies()
                .add_prefix(col + "_")
            )
            dummies["cust_id"] = trx["cust_id"]
            dummies = dummies.groupby("cust_id").sum().reset_index()
            customer_features = customer_features.merge(dummies, on="cust_id", how="left")

    # OUTLET FEATURE
    if "prod_outlet" in trx.columns:
        outlet = trx.groupby("cust_id")["prod_outlet"].mean().reset_index()
        customer_features = customer_features.merge(outlet.rename(columns={"prod_outlet": "prod_outlet_mean"}), on="cust_id", how="left")

    # ORDINAL HEEL
    heel_map = {"<2.5 cm": 0, "2.5-5 cm": 1, "5-8 cm": 2, ">8 cm": 3}
    if "prod_heel" in trx.columns:
        trx["prod_heel_encoded"] = trx["prod_heel"].map(heel_map).fillna(-1)
        heel = trx.groupby("cust_id")["prod_heel_encoded"].mean().reset_index()
        customer_features = customer_features.merge(heel, on="cust_id", how="left")

    # NUEVAS FEATURES IMPORTANTES
    customer_features = add_window_features(trx, customer_features, reference)

    # ratios de retorno / descuento
    gross = customer_features["customer_revenue"] + (customer_features["mean_discount"] * customer_features["n_lines"])
    customer_features["return_rate"] = customer_features["n_returned"] / customer_features["n_lines"].replace(0, np.nan)
    customer_features["discount_rate"] = (customer_features["mean_discount"] * customer_features["n_lines"]) / gross.replace(0, np.nan)

    # cadencia entre compras
    customer_features = add_cadence_features(trx, customer_features)

    # entrega/logística
    if "pack_date" in trx.columns:
        trx["pack_delay_days"] = (trx["pack_date"] - trx["order_date"]).dt.days
        pack = trx.groupby("cust_id")["pack_delay_days"].agg(pack_delay_mean="mean", pack_delay_std="std", pack_delay_max="max").reset_index()
        customer_features = customer_features.merge(pack, on="cust_id", how="left")

    # diversidad
    customer_features = add_diversity_features(trx, customer_features)

    # derivados útiles
    customer_features["avg_revenue_per_order"] = customer_features["customer_revenue"] / customer_features["customer_purchases_number"].replace(0, np.nan)
    customer_features["orders_per_active_month"] = customer_features["customer_purchases_number"] / customer_features["active_months"].replace(0, np.nan)

    return customer_features.replace([np.inf, -np.inf], np.nan).fillna(0)


def tune_classifier(X, y_bin, n_iter=12, folds=3, random_state=42):
    param_space = {
        "iterations": [300, 500, 800, 1000],
        "depth": [4, 5, 6, 7, 8],
        "learning_rate": [0.01, 0.02, 0.03, 0.05],
        "l2_leaf_reg": [1, 3, 5, 7, 9],
        "subsample": [0.7, 0.8, 0.9, 1.0],
    }
    candidates = list(ParameterSampler(param_space, n_iter=n_iter, random_state=random_state))
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)

    best_score = -np.inf
    best_params = None
    for i, params in enumerate(candidates, 1):
        fold_scores = []
        for tr_idx, val_idx in cv.split(X, y_bin):
            model = CatBoostClassifier(**params, loss_function="Logloss", eval_metric="AUC", random_seed=random_state, verbose=False)
            model.fit(X.iloc[tr_idx], y_bin[tr_idx])
            probs = model.predict_proba(X.iloc[val_idx])[:, 1]
            fold_scores.append(roc_auc_score(y_bin[val_idx], probs))
        score = float(np.mean(fold_scores))
        print(f"[CLF {i}/{len(candidates)}] AUC={score:.5f} params={params}")
        if score > best_score:
            best_score = score
            best_params = params
    return best_params, best_score


def tune_regressor(X, y_reg, n_iter=12, folds=3, random_state=42):
    param_space = {
        "iterations": [300, 500, 800, 1000],
        "depth": [4, 5, 6, 7, 8],
        "learning_rate": [0.01, 0.02, 0.03, 0.05],
        "l2_leaf_reg": [1, 3, 5, 7, 9],
        "subsample": [0.7, 0.8, 0.9, 1.0],
    }
    candidates = list(ParameterSampler(param_space, n_iter=n_iter, random_state=random_state))
    reg_labels = make_strat_labels(y_reg, n_bins=10)
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)

    best_score = np.inf
    best_params = None
    for i, params in enumerate(candidates, 1):
        fold_scores = []
        for tr_idx, val_idx in cv.split(X, reg_labels):
            model = CatBoostRegressor(**params, loss_function="MAE", random_seed=random_state, verbose=False)
            model.fit(X.iloc[tr_idx], np.log1p(y_reg[tr_idx]))
            pred = np.expm1(model.predict(X.iloc[val_idx]))
            pred = np.clip(pred, 0, None)
            fold_scores.append(mean_absolute_error(y_reg[val_idx], pred))
        score = float(np.mean(fold_scores))
        print(f"[REG {i}/{len(candidates)}] MAE={score:.5f} params={params}")
        if score < best_score:
            best_score = score
            best_params = params
    return best_params, best_score


def final_revision_checks(X_train_feat, X_test_feat, submission, expected_rows):
    print("\nRevisión final:")
    print(f"Train shape: {X_train_feat.shape}")
    print(f"Test shape: {X_test_feat.shape}")
    print(f"Nulos train: {int(X_train_feat.isna().sum().sum())}")
    print(f"Nulos test: {int(X_test_feat.isna().sum().sum())}")
    print(f"Columnas idénticas train/test: {list(X_train_feat.columns) == list(X_test_feat.columns)}")
    print(f"Filas submission: {len(submission)} (esperado: {expected_rows})")
    print(f"Pred min/max: {submission['revenue_2018_2019'].min():.6f} / {submission['revenue_2018_2019'].max():.6f}")

    assert list(X_train_feat.columns) == list(X_test_feat.columns), "Train/Test columns mismatch"
    assert len(submission) == expected_rows, "Submission row count mismatch"
    assert submission['revenue_2018_2019'].isna().sum() == 0, "Submission contains NaN"


# --- Load data ---
print("Leyendo archivos...")
transactions = pd.read_csv(TRANS_PATH, parse_dates=["order_date", "pack_date"]) if os.path.exists(TRANS_PATH) else None
train_target = pd.read_csv(TRAIN_PATH) if os.path.exists(TRAIN_PATH) else None
test_customers = pd.read_csv(TEST_PATH) if os.path.exists(TEST_PATH) else None
if transactions is None or train_target is None or test_customers is None:
    raise FileNotFoundError("No se encuentran los archivos requeridos.")

# --- Fit-safe setup ---
train_target["cust_id"] = train_target["cust_id"].astype(str)
test_customers["cust_id"] = test_customers["cust_id"].astype(str)
transactions["cust_id"] = transactions["cust_id"].astype(str)

train_ids = set(train_target["cust_id"])
trx_train_only = transactions[transactions["cust_id"].isin(train_ids)].copy()
top_brands = trx_train_only["prod_brand"].value_counts().nlargest(20).index.tolist() if "prod_brand" in trx_train_only.columns else []

print("Construyendo features...")
agg = build_customer_features(transactions, top_brands=top_brands)

X_train = train_target[["cust_id"]].merge(agg, on="cust_id", how="left").fillna(0)
X_test = test_customers[["cust_id"]].merge(agg, on="cust_id", how="left").fillna(0)

y_reg = train_target["revenue_2018_2019"].to_numpy()
y_bin = (y_reg > 0).astype(int)

drop_cols = ["cust_id", "last_order_date", "first_order_date"]
features = [c for c in X_train.columns if c not in drop_cols]

X_train_feat = X_train[features].reset_index(drop=True)
X_test_feat = X_test[features].reset_index(drop=True)

# --- Hyperparameter search ---
clf_params = {"iterations": 1000, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 3, "subsample": 0.8}
reg_params = {"iterations": 1000, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 3, "subsample": 0.8}

if RUN_HYPERPARAM_SEARCH:
    print("Buscando mejores hiperparámetros...")
    tune_df = X_train_feat.copy()
    tune_df["y_bin"] = y_bin
    tune_df["y_reg"] = y_reg
    tune_df = tune_df.sample(frac=TUNE_SAMPLE_FRAC, random_state=RANDOM_STATE).reset_index(drop=True)

    X_tune = tune_df.drop(columns=["y_bin", "y_reg"])
    y_bin_tune = tune_df["y_bin"].to_numpy()
    y_reg_tune = tune_df["y_reg"].to_numpy()

    clf_params, best_auc = tune_classifier(X_tune, y_bin_tune, n_iter=N_ITER_CLF, folds=TUNE_FOLDS, random_state=RANDOM_STATE)
    reg_params, best_mae = tune_regressor(X_tune, y_reg_tune, n_iter=N_ITER_REG, folds=TUNE_FOLDS, random_state=RANDOM_STATE)

    print("Mejor CLF:", clf_params, "AUC:", round(best_auc, 5))
    print("Mejor REG:", reg_params, "MAE:", round(best_mae, 5))

# --- OOF prob_return ---
print("Entrenando CLF OOF...")
skf = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_probs = np.zeros(len(X_train_feat))
test_probs = np.zeros(len(X_test_feat))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_feat, y_bin), 1):
    print(f"Fold CLF {fold}/{NFOLDS}")
    clf = CatBoostClassifier(**clf_params, loss_function="Logloss", eval_metric="AUC", random_seed=RANDOM_STATE, verbose=False)
    clf.fit(X_train_feat.iloc[tr_idx], y_bin[tr_idx])
    oof_probs[val_idx] = clf.predict_proba(X_train_feat.iloc[val_idx])[:, 1]
    test_probs += clf.predict_proba(X_test_feat)[:, 1] / NFOLDS

X_train_feat["prob_return"] = oof_probs
X_test_feat["prob_return"] = test_probs

# --- OOF regression ---
print("Entrenando REG OOF...")
reg_labels = make_strat_labels(y_reg, n_bins=10)
reg_skf = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_preds_reg = np.zeros(len(X_train_feat))

for fold, (tr_idx, val_idx) in enumerate(reg_skf.split(X_train_feat, reg_labels), 1):
    print(f"Fold REG {fold}/{NFOLDS}")
    reg = CatBoostRegressor(**reg_params, loss_function="MAE", random_seed=RANDOM_STATE, verbose=False)
    reg.fit(X_train_feat.iloc[tr_idx], np.log1p(y_reg[tr_idx]))
    oof_preds_reg[val_idx] = reg.predict(X_train_feat.iloc[val_idx])

oof_preds = np.expm1(oof_preds_reg)
oof_preds = np.clip(oof_preds, 0, None)
mae_oof = mean_absolute_error(y_reg, oof_preds)
spearman_oof, _ = spearmanr(y_reg, oof_preds)

print(f"OOF MAE: {mae_oof:.6f}")
print(f"OOF Spearman: {spearman_oof:.6f}")

# --- Final model + submission ---
print("Entrenando modelo final...")
reg_final = CatBoostRegressor(**reg_params, loss_function="MAE", random_seed=RANDOM_STATE, verbose=False)
reg_final.fit(X_train_feat, np.log1p(y_reg))

preds_test = np.expm1(reg_final.predict(X_test_feat))
preds_test = np.clip(preds_test, 0, None)

submission = pd.DataFrame({"cust_id": test_customers["cust_id"].values, "revenue_2018_2019": preds_test})
submission.to_csv(OUTPUT_PATH, index=False)

final_revision_checks(X_train_feat, X_test_feat, submission, expected_rows=len(test_customers))

print("\nSubmission guardada en", OUTPUT_PATH)
print("Resumen final:")
print(f"MAE OOF: {mae_oof:.4f}")
print(f"Spearman OOF: {spearman_oof:.4f}")
print("Parámetros CLF:", clf_params)
print("Parámetros REG:", reg_params)



Leyendo archivos...


/tmp/ipykernel_66740/1207725497.py:204: DtypeWarning: Columns (0: prod_size) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv(TRANS_PATH, parse_dates=["order_date", "pack_date"]) if os.path.exists(TRANS_PATH) else None


Construyendo features a partir de transactions...
drop_duplicates   : 1,063 removed (343,149 remain)
Uniendo features con train/test...
Generando feature prob_return (OOF) con StratifiedKFold...
Fold 1/5
0:	test: 0.7007953	best: 0.7007953 (0)	total: 55.2ms	remaining: 55.1s
100:	test: 0.7234618	best: 0.7234618 (100)	total: 6.64s	remaining: 59.1s
200:	test: 0.7261035	best: 0.7261035 (200)	total: 12.1s	remaining: 48.2s
300:	test: 0.7270800	best: 0.7270958 (292)	total: 17.4s	remaining: 40.4s
400:	test: 0.7276286	best: 0.7276286 (400)	total: 22.8s	remaining: 34s
500:	test: 0.7279494	best: 0.7279629 (477)	total: 28.3s	remaining: 28.2s
600:	test: 0.7280822	best: 0.7281114 (589)	total: 33.8s	remaining: 22.4s
700:	test: 0.7283085	best: 0.7283095 (689)	total: 39.4s	remaining: 16.8s
800:	test: 0.7283952	best: 0.7284618 (779)	total: 44.9s	remaining: 11.2s
900:	test: 0.7283757	best: 0.7284618 (779)	total: 50.5s	remaining: 5.55s
999:	test: 0.7282592	best: 0.7284618 (779)	total: 55.8s	remaining: 0us


In [ ]:
# FAST_SAFE PIPELINE (rápido y estable)
# Objetivo: obtener MAE de referencia sin crashear kernel.

import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostClassifier, CatBoostRegressor

RANDOM_STATE = 42
NFOLDS = 3

TRANS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/transactions_2016_2017.csv"
TRAIN_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/customer_clv_train.csv"
TEST_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/customer_clv_test.csv"
OUTPUT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/submission_fast.csv"


def make_strat_labels(y, n_bins=8):
    y = np.asarray(y)
    labels = np.zeros(len(y), dtype=int)
    pos = y > 0
    if pos.sum() == 0:
        return labels
    try:
        bins = pd.qcut(pd.Series(y[pos]), q=min(n_bins, pos.sum()), labels=False, duplicates="drop")
        labels[pos] = bins.to_numpy(dtype=int) + 1
    except ValueError:
        labels[pos] = 1
    return labels


def build_features_fast(transactions: pd.DataFrame, top_brands):
    trx = transactions.copy()
    trx["cust_id"] = trx["cust_id"].astype(str)
    trx["order_date"] = pd.to_datetime(trx["order_date"], errors="coerce")
    trx["sale_revenue"] = pd.to_numeric(trx["sale_revenue"], errors="coerce").fillna(0.0)
    trx["sale_discount_applied"] = pd.to_numeric(trx.get("sale_discount_applied", 0), errors="coerce").fillna(0.0)

    base = trx.groupby("cust_id").agg(
        revenue_sum=("sale_revenue", "sum"),
        revenue_mean=("sale_revenue", "mean"),
        orders=("sale_id", "nunique"),
        lines=("sale_revenue", "size"),
        products=("prod_id", "nunique"),
        last_order=("order_date", "max"),
        first_order=("order_date", "min"),
        returned=("returned_to_shop_id", lambda x: x.notna().sum())
    ).reset_index()

    ref_date = trx["order_date"].max()
    base["recency_days"] = (ref_date - base["last_order"]).dt.days
    base["tenure_days"] = (ref_date - base["first_order"]).dt.days
    base["return_rate"] = base["returned"] / base["lines"].replace(0, np.nan)
    base["avg_ticket"] = base["revenue_sum"] / base["orders"].replace(0, np.nan)

    # Ventanas rápidas
    for w in [90, 365]:
        cut = ref_date - pd.Timedelta(days=w)
        wdf = trx[trx["order_date"] >= cut]
        agg = wdf.groupby("cust_id").agg(**{
            f"revenue_{w}d": ("sale_revenue", "sum"),
            f"orders_{w}d": ("sale_id", "nunique")
        }).reset_index()
        base = base.merge(agg, on="cust_id", how="left")

    # Top brands (solo contadores)
    if len(top_brands) > 0 and "prod_brand" in trx.columns:
        bc = (trx[trx["prod_brand"].isin(top_brands)]
              .groupby(["cust_id", "prod_brand"]).size()
              .unstack(fill_value=0))
        bc.columns = [f"brand_{c}" for c in bc.columns]
        base = base.merge(bc.reset_index(), on="cust_id", how="left")

    # Categórica compacta
    if "prod_type_1" in trx.columns:
        d = pd.get_dummies(trx["prod_type_1"].fillna("unknown"), prefix="type1")
        d["cust_id"] = trx["cust_id"]
        d = d.groupby("cust_id").sum().reset_index()
        base = base.merge(d, on="cust_id", how="left")

    return base.fillna(0)


start = time.time()
print("[FAST] Leyendo datos...")
transactions = pd.read_csv(TRANS_PATH, parse_dates=["order_date", "pack_date"])
train_target = pd.read_csv(TRAIN_PATH)
test_customers = pd.read_csv(TEST_PATH)

train_target["cust_id"] = train_target["cust_id"].astype(str)
test_customers["cust_id"] = test_customers["cust_id"].astype(str)
transactions["cust_id"] = transactions["cust_id"].astype(str)

train_ids = set(train_target["cust_id"])
trx_train = transactions[transactions["cust_id"].isin(train_ids)]
top_brands = trx_train["prod_brand"].value_counts().head(10).index.tolist() if "prod_brand" in trx_train.columns else []

print("[FAST] Construyendo features...")
feat = build_features_fast(transactions, top_brands)

X_train = train_target[["cust_id"]].merge(feat, on="cust_id", how="left").fillna(0)
X_test = test_customers[["cust_id"]].merge(feat, on="cust_id", how="left").fillna(0)

y = train_target["revenue_2018_2019"].to_numpy()
y_bin = (y > 0).astype(int)

X_train_feat = X_train.drop(columns=["cust_id", "last_order", "first_order"], errors="ignore")
X_test_feat = X_test.drop(columns=["cust_id", "last_order", "first_order"], errors="ignore")

print("[FAST] Entrenando clasificador OOF rápido...")
skf = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_prob = np.zeros(len(X_train_feat))
test_prob = np.zeros(len(X_test_feat))

for i, (tr, va) in enumerate(skf.split(X_train_feat, y_bin), 1):
    clf = CatBoostClassifier(
        iterations=300,
        depth=5,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE + i,
        verbose=False,
    )
    clf.fit(X_train_feat.iloc[tr], y_bin[tr])
    oof_prob[va] = clf.predict_proba(X_train_feat.iloc[va])[:, 1]
    test_prob += clf.predict_proba(X_test_feat)[:, 1] / NFOLDS

X_train_feat = X_train_feat.copy()
X_test_feat = X_test_feat.copy()
X_train_feat["prob_return"] = oof_prob
X_test_feat["prob_return"] = test_prob

print("[FAST] Entrenando regresor OOF rápido...")
labels = make_strat_labels(y, n_bins=8)
reg_cv = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_log = np.zeros(len(X_train_feat))

for i, (tr, va) in enumerate(reg_cv.split(X_train_feat, labels), 1):
    reg = CatBoostRegressor(
        iterations=400,
        depth=6,
        learning_rate=0.05,
        loss_function="MAE",
        random_seed=RANDOM_STATE + i,
        verbose=False,
    )
    reg.fit(X_train_feat.iloc[tr], np.log1p(y[tr]))
    oof_log[va] = reg.predict(X_train_feat.iloc[va])

pred_oof = np.clip(np.expm1(oof_log), 0, None)
mae = mean_absolute_error(y, pred_oof)
print(f"[FAST] OOF MAE: {mae:.6f}")

print("[FAST] Entrenando modelo final...")
reg_final = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="MAE",
    random_seed=RANDOM_STATE,
    verbose=False,
)
reg_final.fit(X_train_feat, np.log1p(y))

pred_test = np.clip(np.expm1(reg_final.predict(X_test_feat)), 0, None)
sub = pd.DataFrame({"cust_id": test_customers["cust_id"], "revenue_2018_2019": pred_test})
sub.to_csv(OUTPUT_PATH, index=False)

elapsed = time.time() - start
print(f"[FAST] Submission: {OUTPUT_PATH}")
print(f"[FAST] Tiempo total: {elapsed/60:.2f} min")



## Celda nueva: hiperparametros rapidos
Ejecuta la siguiente celda para configurar un modo mas rapido.


In [ ]:
# CELDA: hiperparametros rapidos (tradeoff: menos precision, mucho menos tiempo)
# Ejecuta esta celda y usa estos params en tu pipeline principal.

RUN_HYPERPARAM_SEARCH = False
NFOLDS = 3

clf_params = {
    "iterations": 180,
    "depth": 4,
    "learning_rate": 0.08,
    "l2_leaf_reg": 5,
    "subsample": 0.8,
    "bootstrap_type": "Bernoulli",
}

reg_params = {
    "iterations": 240,
    "depth": 5,
    "learning_rate": 0.07,
    "l2_leaf_reg": 5,
    "subsample": 0.8,
    "bootstrap_type": "Bernoulli",
}

print("Modo rapido activado")
print("RUN_HYPERPARAM_SEARCH =", RUN_HYPERPARAM_SEARCH, "| NFOLDS =", NFOLDS)
print("clf_params =", clf_params)
print("reg_params =", reg_params)
